# OSM Commercial + Proximity Distances

Fetches commercial features and adds the distance to the nearest:
**transit stop**, **hospital**, **school**, **park**.

Output: `osm_distances_NNN_YYYY-MM-DD_HH-MM-SS.csv`

---
- Run cells **top to bottom**
- Only edit **Cell 2 — Parameters**
- Kernel: **OSMnx Scraper (Python 3.11)**  
  `Ctrl+Shift+P` → *Notebook: Select Notebook Kernel*

## Cell 1 · Imports & Configuration

In [1]:
import overpy, requests, pandas as pd, math, time, glob
from datetime import datetime

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

COMMERCIAL_TAGS = ["shop", "amenity", "office", "craft", "tourism", "leisure"]
AMENITY_EXCLUDE = {
    "parking", "bench", "waste_basket", "recycling", "post_box",
    "bicycle_parking", "drinking_water", "telephone", "toilets",
    "street_lamp", "fire_station", "school", "place_of_worship", "clock",
}
PROXIMITY_TAGS = {
    "transit":  [
        ("highway",          "bus_stop"),
        ("railway",          "subway_entrance"),
        ("railway",          "station"),
        ("railway",          "tram_stop"),
        ("public_transport", "stop_position"),
        ("amenity",          "bus_station"),
        ("amenity",          "ferry_terminal"),
    ],
    "hospital": [("amenity", "hospital"), ("amenity", "clinic")],
    "school":   [("amenity", "school"), ("amenity", "university"), ("amenity", "college")],
    "park":     [("leisure", "park"), ("leisure", "garden"), ("landuse", "recreation_ground")],
}

print(f"overpy {overpy.__version__} · pandas {pd.__version__} · Ready")


overpy 0.7 · pandas 3.0.2 · Ready


## Cell 2 · Parameters  ← edit here

| Parameter | Description | Default |
|---|---|---|
| `LAT` / `LON` | Origin coordinate | Times Square NYC |
| `WALK_MINUTES` | Scrape radius as walking time | 15 min |
| `WALK_SPEED_M_MIN` | Walking speed m/min | 80 |
| `PROXIMITY_EXTRA_M` | Buffer beyond commercial radius for proximity search | 1500 m |

In [2]:
# ── EDIT THESE ───────────────────────────────────────────────────────────────
LAT               = 40.7589
LON               = -73.9851
WALK_MINUTES      = 15.0
WALK_SPEED_M_MIN  = 80.0
PROXIMITY_EXTRA_M = 1500   # extra meters beyond commercial radius for proximity search
# ─────────────────────────────────────────────────────────────────────────────

RADIUS_M           = WALK_MINUTES * WALK_SPEED_M_MIN
PROXIMITY_RADIUS_M = RADIUS_M + PROXIMITY_EXTRA_M

_existing   = glob.glob("osm_distances_*.csv")
OUTPUT_PATH = f"osm_distances_{len(_existing)+1:03d}_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"

print(f"Origin           : ({LAT}, {LON})")
print(f"Commercial radius: {RADIUS_M:.0f} m")
print(f"Proximity radius : {PROXIMITY_RADIUS_M:.0f} m")
print(f"Output file      : {OUTPUT_PATH}")


Origin           : (40.7589, -73.9851)
Commercial radius: 1200 m
Proximity radius : 2700 m
Output file      : osm_distances_001_2026-04-30_16-38-31.csv


## Cell 3 · Functions — do not edit

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    """Straight-line distance in meters between two coordinates."""
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl  = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def overpass_query(query):
    """POSTs query to Overpass mirrors in order; raises if all fail."""
    last = None
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            return overpy.Overpass().parse_json(r.text)
        except Exception as e:
            last = e
            print(f"  ✗ {ep}: {type(e).__name__}: {e}")
            time.sleep(3)
    raise RuntimeError(f"All Overpass endpoints failed. Last error: {last}")


def coords_of(el):
    """Returns (lat, lon) for a node or way centroid, or None."""
    if hasattr(el, "lat"):
        return float(el.lat), float(el.lon)
    if hasattr(el, "center_lat") and el.center_lat:
        return float(el.center_lat), float(el.center_lon)
    return None


def fetch_commercial(lat, lon, radius):
    """Fetches commercial features within radius meters. Returns DataFrame."""
    lines = (
        [f'node["{t}"](around:{radius:.0f},{lat},{lon});' for t in COMMERCIAL_TAGS] +
        [f'way["{t}"](around:{radius:.0f},{lat},{lon});'  for t in COMMERCIAL_TAGS]
    )
    query = "[out:json][timeout:90];\n(\n    " + "\n    ".join(lines) + "\n);\nout center tags;"

    print(f"[1/2] Fetching commercial features (radius {radius:.0f} m)...")
    result = overpass_query(query)

    rows = []
    for el in list(result.nodes) + list(result.ways):
        c = coords_of(el)
        if not c:
            continue
        elat, elon = c
        tags = el.tags
        pt = pv = None
        for t in COMMERCIAL_TAGS:
            if t in tags:
                v = tags[t]
                if t == "amenity" and v in AMENITY_EXCLUDE:
                    break
                pt, pv = t, v
                break
        if not pt:
            continue
        osm_type = "node" if hasattr(el, "lat") else "way"
        rows.append({
            "osm_id":           el.id,
            "osm_type":         osm_type,
            "lat":              round(elat, 7),
            "lon":              round(elon, 7),
            "distance_m":       round(haversine(lat, lon, elat, elon), 1),
            "primary_tag":      pt,
            "primary_value":    pv,
            "name":             tags.get("name", ""),
            "brand":            tags.get("brand", ""),
            "shop":             tags.get("shop", ""),
            "amenity":          tags.get("amenity", ""),
            "office":           tags.get("office", ""),
            "craft":            tags.get("craft", ""),
            "tourism":          tags.get("tourism", ""),
            "leisure":          tags.get("leisure", ""),
            "cuisine":          tags.get("cuisine", ""),
            "opening_hours":    tags.get("opening_hours", ""),
            "phone":            tags.get("phone", ""),
            "website":          tags.get("website", ""),
            "wheelchair":       tags.get("wheelchair", ""),
            "outdoor_seating":  tags.get("outdoor_seating", ""),
            "takeaway":         tags.get("takeaway", ""),
            "delivery":         tags.get("delivery", ""),
            "level":            tags.get("level", ""),
            "addr_street":      tags.get("addr:street", ""),
            "addr_housenumber": tags.get("addr:housenumber", ""),
            "addr_postcode":    tags.get("addr:postcode", ""),
        })

    df = pd.DataFrame(rows)
    df["has_name"] = df["name"].str.len().gt(0).astype(int)
    df["distance_band"] = pd.cut(
        df["distance_m"],
        bins=[0, 200, 400, 600, 800, 1000, 1200, 9999],
        labels=["0-200m", "200-400m", "400-600m", "600-800m", "800-1000m", "1000-1200m", ">1200m"]
    )
    print(f"  → {len(df)} commercial features found")
    return df


def fetch_proximity_pois(lat, lon, radius):
    """
    Fetches transit stops, hospitals, schools and parks in a single Overpass call.
    Returns: {category: [(lat, lon, name), ...]}
    """
    tag_to_cat = {}
    lines = []
    for cat, pairs in PROXIMITY_TAGS.items():
        for key, val in pairs:
            tag_to_cat[(key, val)] = cat
            lines.append(f'node["{key}"="{val}"](around:{radius:.0f},{lat},{lon});')
            lines.append(f'way["{key}"="{val}"](around:{radius:.0f},{lat},{lon});')

    query = "[out:json][timeout:90];\n(\n    " + "\n    ".join(lines) + "\n);\nout center tags;"

    print(f"[2/2] Fetching proximity POIs (radius {radius:.0f} m) — waiting 5 s...")
    time.sleep(5)
    result = overpass_query(query)

    pois = {cat: [] for cat in PROXIMITY_TAGS}
    for el in list(result.nodes) + list(result.ways):
        c = coords_of(el)
        if not c:
            continue
        elat, elon = c
        name = el.tags.get("name", "")
        for (key, val), cat in tag_to_cat.items():
            if el.tags.get(key) == val:
                pois[cat].append((elat, elon, name))
                break

    for cat, items in pois.items():
        print(f"  {cat:10s}: {len(items)} POIs found")
    return pois


def add_distances(df, pois):
    """
    Adds 8 columns to df:
      dist_transit_m, nearest_transit
      dist_hospital_m, nearest_hospital
      dist_school_m,   nearest_school
      dist_park_m,     nearest_park
    """
    for cat, poi_list in pois.items():
        dists, names = [], []
        for _, row in df.iterrows():
            if not poi_list:
                dists.append(None)
                names.append("")
                continue
            best_d, best_n = math.inf, ""
            for plat, plon, pname in poi_list:
                d = haversine(row["lat"], row["lon"], plat, plon)
                if d < best_d:
                    best_d, best_n = d, pname
            dists.append(round(best_d, 1))
            names.append(best_n)
        df[f"dist_{cat}_m"]    = dists
        df[f"nearest_{cat}"]   = names
        filled = [d for d in dists if d is not None]
        if filled:
            print(f"  dist_{cat}_m → min {min(filled):.0f} m · avg {sum(filled)/len(filled):.0f} m · max {max(filled):.0f} m")
        else:
            print(f"  WARNING: no {cat} POIs — increase PROXIMITY_EXTRA_M")
    return df


print("All functions defined — ready to run Cell 4.")


All functions defined — ready to run Cell 4.


## Cell 4 · Run

Runs both API calls and saves `osm_distances_*.csv` with all proximity columns.  
Expect **30–90 seconds** depending on server load.

In [4]:
# ── Step 1: commercial features ───────────────────────────────────────────────
df = fetch_commercial(LAT, LON, RADIUS_M)
assert len(df) > 0, "No commercial features found — check coordinates."

# ── Step 2: proximity POIs (transit / hospital / school / park) ───────────────
pois = fetch_proximity_pois(LAT, LON, PROXIMITY_RADIUS_M)

# ── Step 3: compute nearest distance for every feature ────────────────────────
print("\nComputing nearest distances...")
df = add_distances(df, pois)

# ── Step 4: sort and save ─────────────────────────────────────────────────────
df.sort_values("distance_m", inplace=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"\n>>> SAVED: {OUTPUT_PATH}")
print(f"    Rows   : {len(df)}")
print(f"    Columns: {len(df.columns)}")
print(f"    List   : {list(df.columns)}")


[1/2] Fetching commercial features (radius 1200 m)...
  → 3704 commercial features found
[2/2] Fetching proximity POIs (radius 2700 m) — waiting 5 s...
  transit   : 1438 POIs found
  hospital  : 99 POIs found
  school    : 148 POIs found
  park      : 1043 POIs found

Computing nearest distances...
  dist_transit_m → min 0 m · avg 62 m · max 470 m
  dist_hospital_m → min 0 m · avg 242 m · max 1062 m
  dist_school_m → min 0 m · avg 213 m · max 704 m
  dist_park_m → min 0 m · avg 122 m · max 389 m

>>> SAVED: osm_distances_001_2026-04-30_16-38-31.csv
    Rows   : 3704
    Columns: 37
    List   : ['osm_id', 'osm_type', 'lat', 'lon', 'distance_m', 'primary_tag', 'primary_value', 'name', 'brand', 'shop', 'amenity', 'office', 'craft', 'tourism', 'leisure', 'cuisine', 'opening_hours', 'phone', 'website', 'wheelchair', 'outdoor_seating', 'takeaway', 'delivery', 'level', 'addr_street', 'addr_housenumber', 'addr_postcode', 'has_name', 'distance_band', 'dist_transit_m', 'nearest_transit', 'dist

## Cell 5 · Explore Results

In [5]:
df.head(10)

,osm_id,osm_type,lat,lon,distance_m,primary_tag,primary_value,name,brand,shop,...,has_name,distance_band,dist_transit_m,nearest_transit,dist_hospital_m,nearest_hospital,dist_school_m,nearest_school,dist_park_m,nearest_park
2176,10228962372,node,40.758980,-73.985221,13.5,leisure,outdoor_seating,,,,...,0,0-200m,62.6,,421.5,Manhattan Gastroenterology – Midtown Manhattan,329.8,Repertory Company High School for Theatre Arts,25.3,
3556,1118420739,way,40.758759,-73.985146,16.1,leisure,garden,,,,...,0,0-200m,70.4,,399.2,Manhattan Gastroenterology – Midtown Manhattan,304.6,Repertory Company High School for Theatre Arts,0.0,
258,2435110791,node,40.758759,-73.985146,16.1,tourism,artwork,George M. Cohan,,,...,1,0-200m,70.4,,399.2,Manhattan Gastroenterology – Midtown Manhattan,304.6,Repertory Company High School for Theatre Arts,0.0,
325,2639531695,node,40.759057,-73.984989,19.8,tourism,artwork,Francis P. Duffy,,,...,1,0-200m,47.2,,415.3,Manhattan Gastroenterology – Midtown Manhattan,335.5,Repertory Company High School for Theatre Arts,35.6,
3510,796338420,way,40.758949,-73.985385,24.6,shop,newsagent,,,newsagent,...,0,0-200m,73.2,,428.4,Manhattan Gastroenterology – Midtown Manhattan,329.1,Repertory Company High School for Theatre Arts,29.1,
1814,7447580897,node,40.758881,-73.985403,25.6,tourism,information,Walk NYC,,,...,1,0-200m,80.1,,424.0,Manhattan Gastroenterology – Midtown Manhattan,322.1,Repertory Company High School for Theatre Arts,25.5,
3110,258619240,way,40.759153,-73.984927,31.7,shop,ticket,TKTS,TKTS,ticket,...,1,0-200m,35.6,,420.4,Manhattan Gastroenterology – Midtown Manhattan,345.6,Repertory Company High School for Theatre Arts,41.6,
3698,1413533223,way,40.759153,-73.984927,31.7,tourism,viewpoint,,,,...,0,0-200m,35.6,,420.4,Manhattan Gastroenterology – Midtown Manhattan,345.6,Repertory Company High School for Theatre Arts,41.6,
310,2613887631,node,40.758865,-73.985491,33.1,shop,clothes,American Eagle Outfitters,American Eagle Outfitters,clothes,...,1,0-200m,86.0,,428.0,Manhattan Gastroenterology – Midtown Manhattan,322.1,Repertory Company High School for Theatre Arts,31.3,
755,3595084281,node,40.759125,-73.985377,34.2,shop,gift,Grand Slam New York,,gift,...,1,0-200m,58.5,,442.3,Manhattan Gastroenterology – Midtown Manhattan,332.8,Public School 212,45.0,


In [6]:
prox = ["dist_transit_m", "dist_hospital_m", "dist_school_m", "dist_park_m"]
df[prox].describe().round(1)


,dist_transit_m,dist_hospital_m,dist_school_m,dist_park_m
count,3704.0,3704.0,3704.0,3704.0
mean,61.8,242.4,213.0,121.5
std,43.0,119.0,109.0,80.1
min,0.0,0.0,0.0,0.0
25%,27.7,158.2,130.9,59.7
50%,53.5,227.3,192.6,113.8
75%,88.3,319.8,285.0,175.4
max,469.8,1061.9,704.4,389.0


In [7]:
df[["name", "primary_value", "dist_transit_m", "nearest_transit",
    "dist_hospital_m", "nearest_hospital",
    "dist_school_m",   "nearest_school",
    "dist_park_m",     "nearest_park"]].sort_values("dist_transit_m").head(20)


,name,primary_value,dist_transit_m,nearest_transit,dist_hospital_m,nearest_hospital,dist_school_m,nearest_school,dist_park_m,nearest_park
3083,Port Authority Bus Terminal,bus_station,0.0,Port Authority Bus Terminal,145.5,CityMD,181.0,Holy Cross School,197.8,Lobby Garden
3568,,shelter,0.9,West 34th Street/Broadway,185.2,L.A. FUE Hair New York,31.9,Mercy University,74.0,
679,Tim Hortons,cafe,1.3,New York Penn Station,186.0,CityMD,118.8,Laguardia Community College,78.4,
777,Citi Bike - E 53 St & Madison Ave,bicycle_rental,2.4,53rd Street–5th Avenue,407.3,Russak Dermatology Clinic,86.5,Laboratory Institute of Merchandising,113.1,Paley Park
3390,Zaro's Family Backery,bakery,2.7,Grand Central 43rd Street / Vanderbilt Avenue ...,232.7,Manhattan Restorative Health Sciences,413.8,SUNY State College of Optometry,30.3,
2522,Citibank,bank,2.9,,114.5,Elevate Egg Donors and Surrogates,129.7,Stella and Charles Guttman Community College,49.0,
86,USA Brooklyn Delicatessen,restaurant,2.9,,27.6,New York Cardiac Diagnostic Center,226.5,Public School 69,222.8,
111,LA Fonda Del Sol,restaurant,3.1,Grand Central Terminal,269.2,Manhattan Restorative Health Sciences,424.9,SUNY State College of Optometry,64.6,
3569,,newsagent,3.2,,191.5,L.A. FUE Hair New York,44.4,Mercy University,70.6,
2893,Citi Bike - W 35 St & 8 Ave,bicycle_rental,3.4,,140.2,CityMD,148.8,Laguardia Community College,94.8,
